# Step 1 — Deploy-image probe (offline)

The open-loop replay proved the **model + pipeline reproduce training actions** from training
frames. So the standstill comes from the **deploy observations**. This notebook feeds the
**actual deploy camera images** (`recorded_obs/`) into the same pipeline and checks whether the
output **collapses to near-home** (reproducing the standstill) — then ablates the likely image
causes (RGB/BGR, the server's extra 256 down-resize).

**Setup notes**
- Runs on the **GPU server**: copy the `recorded_obs/` folder (from the client machine) next to
  this notebook, or set `RECORDED_DIR`.
- `record_obs` does **not** log `observation.state`, so we pair the deploy images with an
  **in-distribution proxy state** (dataset episode-0 start). This isolates the *image* effect.
  (Step 2 will log the real state and pair them exactly.)

**Read-out:** if deploy images give a near-constant / near-home action while dataset images (same
fixed state) give a varied/reaching action ⇒ the **deploy image content is OOD** to the model.

In [ ]:
import os, glob
import numpy as np
import torch
import matplotlib.pyplot as plt
from PIL import Image

from lerobot.datasets.lerobot_dataset import LeRobotDataset
from lerobot.policies.factory import get_policy_class, make_pre_post_processors

MODEL = "di-techinnova/smolvla-pouring-0.1"
DATASET = "di-techinnova/so-arm-101-pouring-0.2"
RECORDED_DIR = "/home/trietlm/lerobot/recorded_obs"           # copy from the client machine to the GPU server
TASK = "Pour from orange cup into blue cup."
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
JOINTS = ["sh_pan", "sh_lift", "elbow", "wr_flex", "wr_roll", "grip"]
print("device:", DEVICE, "| recorded_obs exists:", os.path.isdir(RECORDED_DIR))

## 1. Load policy + the server's pre/post processors

In [ ]:
policy = get_policy_class("smolvla").from_pretrained(MODEL).to(DEVICE).eval()
pre, post = make_pre_post_processors(
    policy.config,
    pretrained_path=MODEL,
    preprocessor_overrides={
        "device_processor": {"device": DEVICE},
        "rename_observations_processor": {"rename_map": {}},
    },
    postprocessor_overrides={"device_processor": {"device": DEVICE}},
)
IMG_KEYS = [k for k in policy.config.input_features if k.startswith("observation.images.")]
print("model image inputs:", IMG_KEYS)

## 2. Helpers + in-distribution proxy state

In [ ]:
# state: (6,) tensor; imgs: {model_image_key: (3,H,W) float[0,1]}. Returns unnorm action (6,).
def run_action(state, imgs, task=TASK):
    obs = {"observation.state": state.unsqueeze(0)}
    for k, im in imgs.items():
        obs[k] = im.unsqueeze(0)
    obs["task"] = task
    policy.reset()
    processed = pre(obs)
    with torch.no_grad():
        chunk = policy.predict_action_chunk(processed)
    if chunk.ndim != 3:
        chunk = chunk.unsqueeze(0)
    return post(chunk[:, 0, :]).detach().cpu().numpy().ravel()[:6]

# PNG was saved as cv2.imwrite(RGB2BGR(frame)) -> opening as RGB recovers the exact frame.
def load_deploy_img(cam, fname):
    arr = np.asarray(Image.open(os.path.join(RECORDED_DIR, "images", cam, fname)).convert("RGB"))
    return torch.from_numpy(arr).permute(2, 0, 1).float() / 255.0  # (3,H,W) [0,1]

# deploy camera folders -> the two physical cameras (model keys camera1/camera2)
deploy_cams = sorted(os.listdir(os.path.join(RECORDED_DIR, "images")))
print("deploy camera folders:", deploy_cams)
cam_files = sorted(os.listdir(os.path.join(RECORDED_DIR, "images", deploy_cams[0])))
deploy_steps = cam_files[:: max(1, len(cam_files) // 6)][:6]
print("deploy frames sampled:", deploy_steps)

# proxy state = dataset episode-0 start (in-distribution)
ds = LeRobotDataset(DATASET, video_backend="pyav")  # torchcodec/FFmpeg broken on server
try:
    f0 = int(ds.episode_data_index["from"][0]); t0 = int(ds.episode_data_index["to"][0])
except Exception:
    f0, t0 = 0, 450
proxy_state = ds[f0]["observation.state"]
print("proxy state (dataset start):", np.round(proxy_state.numpy(), 1))

## 3. Main test — deploy images vs dataset images, with the **same** proxy state

`deploy`   = real recorded images + proxy state.
`ds_fixed` = dataset images (different frames) + **same** proxy state — control that the model
*does* respond to in-distribution image content when the state is held fixed.

In [ ]:
def imgs_from_deploy(fname):
    out = {}
    for cam, key in zip(deploy_cams, ["observation.images.camera1", "observation.images.camera2"]):
        if key in IMG_KEYS:
            out[key] = load_deploy_img(cam, fname)
    return out

def imgs_from_dataset(gi):
    item = ds[gi]
    out = {}
    for k in IMG_KEYS:
        if k in item:
            im = item[k]
            if im.dtype == torch.uint8:
                im = im.float() / 255.0
            out[k] = im
    return out

print("=== DEPLOY images + proxy state ===")
out_deploy = []
for fn in deploy_steps:
    a = run_action(proxy_state, imgs_from_deploy(fn))
    out_deploy.append(a); print(f"  {fn}: {np.round(a,1)}")
out_deploy = np.stack(out_deploy)

print("\n=== DATASET images (varying frame) + SAME proxy state ===")
ds_frames = [f0 + o for o in [0, 75, 150, 270, 350] if f0 + o < t0]
out_dsfix = []
for gi in ds_frames:
    a = run_action(proxy_state, imgs_from_dataset(gi))
    out_dsfix.append(a); print(f"  frame {gi-f0:3d}: {np.round(a,1)}")
out_dsfix = np.stack(out_dsfix)

## 4. Verdict — does deploy-image output collapse while dataset-image output varies?

In [ ]:
dep_var = out_deploy.std(0)
ds_var = out_dsfix.std(0)
print(f"DEPLOY  output std/joint : {np.round(dep_var,1)}  (mean {dep_var.mean():.1f})")
print(f"DATASET output std/joint : {np.round(ds_var,1)}  (mean {ds_var.mean():.1f})")
print(f"DEPLOY  output mean      : {np.round(out_deploy.mean(0),1)}")
print(f"DATASET output mean      : {np.round(out_dsfix.mean(0),1)}")
print()
if dep_var.mean() < 0.4 * ds_var.mean():
    print("=> DEPLOY images give a near-CONSTANT action while dataset images vary.")
    print("   The deploy image CONTENT is OOD to the model -> camera domain gap.")
    print("   Continue to the ablations (RGB/BGR, 256-resize) to narrow it down.")
else:
    print("=> Deploy images DO drive variation. Image content is probably not the sole cause;")
    print("   suspect the server's 256 down-resize (ablation B) or the state pipeline (Step 2).")

## 5. Ablation A — channel order (RGB vs BGR)

In [ ]:
def imgs_from_deploy_bgr(fname):
    out = {}
    for cam, key in zip(deploy_cams, ["observation.images.camera1", "observation.images.camera2"]):
        if key in IMG_KEYS:
            out[key] = load_deploy_img(cam, fname).flip(0)  # swap R<->B channels
    return out

print("deploy frame -> [RGB action] vs [BGR-swapped action]")
for fn in deploy_steps[:3]:
    a_rgb = run_action(proxy_state, imgs_from_deploy(fn))
    a_bgr = run_action(proxy_state, imgs_from_deploy_bgr(fn))
    print(f"  {fn}: RGB {np.round(a_rgb,1)} | BGR {np.round(a_bgr,1)}")
print("\nIf BGR suddenly produces a reaching/varied action -> deploy camera color order is wrong.")

## 6. Ablation B — the server's extra 256 down-resize

In [ ]:
import torch.nn.functional as F

def downup(img, size=256):
    x = F.interpolate(img.unsqueeze(0), size=(size, size), mode="bilinear", align_corners=False)
    return x.squeeze(0)

print("dataset frame -> [native res action] vs [256 down-resized action]   (state = matching frame)")
for gi in ds_frames[:4]:
    item = ds[gi]
    st = item["observation.state"]
    native = imgs_from_dataset(gi)
    re256 = {k: downup(v) for k, v in native.items()}
    a_nat = run_action(st, native)
    a_256 = run_action(st, re256)
    gt = item["action"].numpy().ravel()[:6]
    print(f"  frame {gi-f0:3d}: native {np.round(a_nat,1)} | 256 {np.round(a_256,1)} | gt {np.round(gt,1)}")
print("\nIf 256 collapses toward home while native matches gt -> the server's down-resize is a cause.")

## 7. Direct image diff — deploy vs dataset (episode-0 start)

In [ ]:
dep0 = load_deploy_img(deploy_cams[0], deploy_steps[0])           # (3,H,W) [0,1]
ds0 = imgs_from_dataset(f0)["observation.images.camera1"]
print("deploy cam1 shape:", tuple(dep0.shape), "| dataset cam1 shape:", tuple(ds0.shape))

fig, axs = plt.subplots(1, 3, figsize=(15, 4))
axs[0].imshow(dep0.permute(1, 2, 0).numpy()); axs[0].set_title("deploy cam1 (start)"); axs[0].axis("off")
axs[1].imshow(ds0.permute(1, 2, 0).numpy()); axs[1].set_title("dataset cam1 (start)"); axs[1].axis("off")
for ci, c in enumerate(["r", "g", "b"]):
    axs[2].hist(dep0[ci].numpy().ravel(), bins=40, alpha=.4, color=c, histtype="step", lw=2, label=f"deploy-{c}")
    axs[2].hist(ds0[ci].numpy().ravel(), bins=40, alpha=.4, color=c, histtype="stepfilled", label=f"ds-{c}")
axs[2].set_title("per-channel intensity"); axs[2].legend(fontsize=7)
plt.tight_layout(); plt.show()
print("deploy mean RGB:", np.round(dep0.mean((1, 2)).numpy(), 3), "| dataset mean RGB:", np.round(ds0.mean((1, 2)).numpy(), 3))

## How to read

- **§4 deploy std ≪ dataset std** → deploy images don't drive the model: a **camera domain gap**
  (framing/FOV/mount/lighting/background differ from training). Fix the rig to match the dataset,
  or collect a few demos with the current rig and fine-tune.
- **§5 BGR fixes it** → a color-order bug in the deploy camera path.
- **§6 256 collapses but native matches gt** → the server's pre-resize to `[3,256,256]` destroys
  the wrist-cam detail the policy needs; feed higher-res to the policy.
- **§7** quantifies brightness/color/scene differences directly.

If all image tests look fine, the remaining suspect is the **state** (calibration drift) — that is
**Step 2** (log real deploy state and pair exactly).

## 8. Ablation C — JPEG q90 transport (config `image_compress_enable=true`)

The recorded PNGs are **raw**, but at deploy the model saw **JPEG q90** frames. Re-check the matched dataset frames raw-vs-jpeg, and the deploy frames raw-vs-jpeg.

In [ ]:
JPEG_QUALITY = 90  # matches deploy config image_compress_quality
import cv2
def jpeg_roundtrip(img_chw, quality=JPEG_QUALITY):
    arr = (img_chw.detach().cpu().permute(1, 2, 0).numpy() * 255.0).clip(0, 255).astype(np.uint8)
    ok, enc = cv2.imencode('.jpg', arr, [int(cv2.IMWRITE_JPEG_QUALITY), int(quality)])
    dec = cv2.imdecode(enc, cv2.IMREAD_COLOR)
    return torch.from_numpy(dec).permute(2, 0, 1).float() / 255.0

print('dataset matched frame -> [native] vs [jpeg q90] vs gt')
for gi in ds_frames[:4]:
    item = ds[gi]; st = item['observation.state']
    nat = imgs_from_dataset(gi)
    jpg = {k: jpeg_roundtrip(v) for k, v in nat.items()}
    gt = item['action'].numpy().ravel()[:6]
    print(f'  frame {gi-f0:3d}: native {np.round(run_action(st,nat),1)} | jpeg {np.round(run_action(st,jpg),1)} | gt {np.round(gt,1)}')
print('deploy frame -> [raw] vs [jpeg q90]   (proxy state)')
for fn in deploy_steps[:3]:
    raw = imgs_from_deploy(fn)
    jpg = {k: jpeg_roundtrip(v) for k, v in raw.items()}
    print(f'  {fn}: raw {np.round(run_action(proxy_state,raw),1)} | jpeg {np.round(run_action(proxy_state,jpg),1)}')
print('If jpeg degrades the matched dataset output toward home -> JPEG transport contributes.')